In [3]:
TEST = """
rozdział siedemnasty nowy [-zwierzchnik
od-] {+zwierzchnikod+} początku roku szkolnego wiedziano powszechnie że surowy inspektor nazywany w owe czasy rektorem został od obowiązków swych usunięty i że pełni je tylko czasowo czekając na przybycie następcy o powodach usunięcia różne między chłopcami krążyły wieści twierdzono przeważnie że to była kara za zbytnią surowość jakiś knot rozpieszczony przez matkę nigdy palcem przez nikogo nie tknięty jedynaczek po otrzymaniu z rozkazu inspektora dwudziestu [-rózeg-] {+ruzek+} miał ciężko rozchorować się podobno nawet [-umrzeć
osobistości-] {+umrzećosobistości+} ofiary nikt nie umiał dokładnie wskazać sam jednak fakt narodzenia się tej wieści i jej prawdopodobieństwa był bardzo wymowny być może zresztą że sprawa przedstawiała się nierównie [-prościej
inspektor-] {+prościejinspektor+} człowiek stary już mógł [-był-] {+by+} lata
"""

In [4]:
REPLACE = r"\[-([^-]*)-\] \{\+([^\+]*)\+\}"
INSERT = r"\{\+([^\+]*)\+\}"
DELETE = r"\[-([^-]*)-\]"

In [5]:
def process_wdiff_output(text):
    text = text.replace("\n", " ")
    

In [6]:
import re

def get_confusables():
    CONFUSABLES_RAW = {
        "e": ["ę", "e"],
        "ę": ["ę", "e", "en", "em"]
    }
    CONFUSEABLES_SYM = """
    sz ż rz
    t d
    p b
    dź ć
    ą om on oł
    w f
    s z
    ź ś
    dz c
    cz dż drz
    h ch
    k g
    """
    for cfs in CONFUSEABLES_SYM.split("\n"):
        if cfs == "":
            continue
        cfsitems = cfs.strip().split(" ")
        for cfsitem in cfsitems:
            if cfsitem == '':
                continue
            CONFUSABLES_RAW[cfsitem] = cfsitems
    return {x: f"({'|'.join(CONFUSABLES_RAW[x])})" for x in CONFUSABLES_RAW}

def eq_mod_confusables(a, b):
    a = a.replace(" ", "")
    b = b.replace(" ", "")
    confusables = get_confusables()
    search_re = rf"*({'|'.join(sorted(confusables.keys(), key=len, reverse=True))})*"
    


def eqnospace(a, b):
    return a.replace(" ", "") == b.replace(" ", "")

def get_replacement(text):
    rep = re.match(REPLACE, text)
    if rep:
        a = rep.group(1)
        b = rep.group(2)
        if eqnospace(a, b):
            return a
        elif a.startswith("z ") and b.startswith("s") and a[2].replace(" ", "") == b[1].replace(" ", ""):
            return a
        


In [7]:
get_replacement("[-zwierzchnik od-] {+zwierzchnikod+}")

'zwierzchnik od'

In [28]:
confusables = get_confusables()
search_re = re.compile(rf"({'|'.join(sorted(confusables.keys(), key=len, reverse=True))})")
example = "niszaczeszy"
output = example

last_end = 0
pre = ""
result = ""
for match in re.finditer(search_re, example):
    start, end = match.span()
    if start != last_end:
        result = result + example[last_end:start]
    result = result + confusables[example[start:end]]
    last_end = end
result = result + example[last_end:]
    # str = re.sub(pattern, repl, str)
    # match = re.search(pattern, str)

In [30]:
result

'ni(sz|ż|rz)a(cz|dż|drz)(ę|e)(sz|ż|rz)y'